# 🔌 Large-Scale EV Smart Charging: A Four-Pillar AI Project
## IEOR E4010 — Artificial Intelligence for OR and Financial Engineering
### Columbia University · Spring 2026 · Final Project

---

**Objective:** Minimize electricity costs for a fleet depot with 20 EVs, a 150 kW transformer limit, and V2G capability — using ML forecasting, DL sequence models, RL control, and an Agentic AI orchestrator.

**How to use this notebook:**
1. Run cells top-to-bottom
2. Sections marked ★ **TODO** are where students implement key components
3. All code runs as-is — student TODOs improve upon working baselines

## 0. Setup & Configuration

In [ ]:
# Install dependencies (uncomment if running in Colab or fresh environment)
# !pip install -r requirements.txt

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Project modules
from config import Config, DEFAULT_CONFIG
from data_utils import (
    generate_synthetic_prices,
    load_realistic_prices,
    get_daily_price_curve,
    generate_ev_schedules,
    train_test_split_prices,
    plot_price_profile,
    plot_ev_schedules,
)

# Set random seed for reproducibility
cfg = Config()
np.random.seed(cfg.seed)
print(cfg.summary())

## 1. Data Generation

### 1a. Electricity Prices

We provide **two options**:
- **Synthetic (default):** CAISO-style price patterns generated offline. No internet needed.
- **Realistic (optional):** Real wholesale prices from US ISOs via `gridstatus`.

The synthetic data captures three key patterns:
1. **Overnight valley** — cheap power at 2–5 AM
2. **Solar duck curve** — midday dip from solar surplus
3. **Evening peak** — expensive power at 5–9 PM (this is when V2G pays off)

In [ ]:
# ==============================================================
# Option A: Synthetic prices (default — works fully offline)
# ==============================================================
prices_df = generate_synthetic_prices(cfg)
print(f"Generated {len(prices_df)} hourly price records over {cfg.price.num_days_history} days")
print(f"Price stats: mean=${prices_df['price_mwh'].mean():.1f}, "
      f"min=${prices_df['price_mwh'].min():.1f}, "
      f"max=${prices_df['price_mwh'].max():.1f}/MWh")

# ==============================================================
# Option B: Realistic prices (requires internet + gridstatus)
# Uncomment below to use real CAISO data instead:
# ==============================================================
# # !pip install gridstatus
# prices_df = load_realistic_prices(
#     iso="CAISO",
#     start_date="2024-01-01",
#     end_date="2024-12-31",
#     node="TH_SP15_GEN-APND",
# )

In [ ]:
# Visualize price patterns
plot_price_profile(prices_df, num_days=10)

### 1b. EV Fleet Schedules

In [ ]:
# Generate stochastic EV arrival/departure schedules
schedules = generate_ev_schedules(cfg)

print("Sample EV schedules:")
for ev in schedules[:5]:
    print(f"  {ev}")
print(f"  ... ({len(schedules)} total)")

In [ ]:
# Visualize fleet schedules (Gantt chart — color = charging urgency)
plot_ev_schedules(schedules, cfg)

### 1c. Train/Test Split & Day Extraction

In [ ]:
# Chronological train/test split (no data leakage)
train_df, test_df = train_test_split_prices(prices_df, cfg)

# Extract one day's price curve for simulation (15-min resolution)
price_curve = get_daily_price_curve(prices_df, day_index=0, cfg=cfg)
print(f"Day 0 price curve: {price_curve.shape[0]} steps, "
      f"range ${price_curve.min():.1f}–${price_curve.max():.1f}/MWh")

plt.figure(figsize=(12, 4))
steps = np.arange(len(price_curve))
hours = (cfg.time.start_hour + steps * cfg.time.dt_hours) % 24
plt.plot(hours, price_curve, 'b-', linewidth=1.5)
plt.xlabel("Hour of Day")
plt.ylabel("Price ($/MWh)")
plt.title("Day 0 Price Curve (15-min resolution)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Baseline Strategies (Heuristics)

Before applying AI, we establish baselines with simple rule-based strategies.
These answer: *"How well can you do without any intelligence at all?"*

- **ASAP**: Charge every EV at full power immediately upon arrival
- **ALAP**: Wait as long as possible, then charge at full power
- **Round Robin**: Rotate charging slots to respect the transformer limit

In [ ]:
from heuristics import run_all_heuristics, run_heuristic
from environment import make_env, plot_episode_results

# Run all heuristics on the SAME scenario (same schedules, same prices)
heuristic_results = run_all_heuristics(
    cfg, schedules=schedules, price_curve=price_curve, verbose=True
)

In [ ]:
# Visualize the Round Robin strategy in detail
env_rr = make_env(cfg, schedules=schedules, price_curve=price_curve)
_ = run_heuristic(env_rr, strategy="round_robin")
plot_episode_results(env_rr, title="Round Robin Baseline")

## 3. LP Optimal Solution (Gold Standard)

The LP optimizer solves the problem to **global optimality** — but it requires
**perfect foresight** (it knows all future prices, arrivals, and SoC values).

No real-time controller can access this information, so the LP serves as a
**theoretical ceiling**. The gap between LP optimal and any online method
represents the unavoidable *cost of uncertainty*.

For IEOR students: this is a linear program with coupling constraints
(transformer limit). The formulation is in `optimizer.py`.

In [ ]:
from optimizer import solve_optimal_schedule, plot_optimal_schedule

# Solve LP with V2G enabled
lp_result = solve_optimal_schedule(
    schedules, price_curve, cfg, allow_v2g=True, verbose=True
)

In [ ]:
# Visualize LP optimal schedule
plot_optimal_schedule(lp_result, price_curve, cfg)

## 4. Pillar 1: Machine Learning — Price Forecasting

### ★ STUDENT TODO: Improve the feature engineering pipeline

The ML forecaster uses XGBoost to predict next-hour prices from tabular features.
The current implementation includes basic features — students improve them.

**Your task:**
1. Review `engineer_features()` in `ml_forecaster.py`
2. Add new features (see suggestions in the code comments)
3. Tune XGBoost hyperparameters
4. Analyze feature importances — which features matter most?

In [ ]:
from ml_forecaster import (
    engineer_features,
    train_ml_models,
    evaluate_ml_models,
    get_feature_importance,
    plot_ml_results,
    plot_feature_importance,
)

# Engineer features
train_feat = engineer_features(train_df, cfg)
test_feat = engineer_features(test_df, cfg)
feature_cols = [c for c in train_feat.columns if c not in {'datetime','price_mwh','target','day_index'}]
print(f"Feature count: {len(feature_cols)}")
print(f"Train: {len(train_feat)} samples, Test: {len(test_feat)} samples")

In [ ]:
# Train ML models (XGBoost + Random Forest)
ml_models = train_ml_models(train_feat, cfg)

# Evaluate on test set
ml_results = evaluate_ml_models(ml_models, test_feat)

In [ ]:
# Visualize ML forecasting results
plot_ml_results(ml_results, test_feat, num_days=7)

In [ ]:
# Feature importance analysis
imp_df = get_feature_importance(ml_models, top_n=15)
print(imp_df.to_string(index=False))
plot_feature_importance(imp_df)

## 5. Pillar 2: Deep Learning — LSTM Sequence Forecasting

### ★ STUDENT TODO: Improve the LSTM architecture and training

The LSTM learns price patterns from raw sequential data.
Unlike XGBoost, it learns its own feature representations from the sequence.

**Your task:**
1. Review `PriceLSTM` and `train_lstm()` in `dl_forecaster.py`
2. Experiment with hidden_size, num_layers, sequence_length
3. Compare LSTM vs. XGBoost — is DL actually better on this data?

In [ ]:
from dl_forecaster import (
    train_lstm,
    evaluate_lstm,
    plot_dl_results,
)

train_prices = train_df["price_mwh"].values
test_prices = test_df["price_mwh"].values

In [ ]:
# Train LSTM (reduce epochs for faster iteration; increase for better results)
cfg.forecast.lstm_epochs = 30

lstm_dict = train_lstm(
    train_prices,
    val_prices=test_prices[:2000],
    cfg=cfg,
    verbose=True,
)

In [ ]:
# Evaluate LSTM on test set
lstm_results = evaluate_lstm(lstm_dict, test_prices)

In [ ]:
# Head-to-head comparison: ML vs DL
plot_dl_results(lstm_results, ml_results=ml_results, history=lstm_dict["history"])

# Comparison table
print("\n" + "="*55)
print("ML vs. DL Comparison")
print("="*55)
print(f"{'Model':<15} {'MAE ($/MWh)':<15} {'RMSE ($/MWh)':<15} {'R²':<10}")
print("-"*55)
for name, res in ml_results.items():
    print(f"{name:<15} ${res['mae']:<14.2f} ${res['rmse']:<14.2f} {res['r2']:<10.4f}")
print(f"{'LSTM':<15} ${lstm_results['mae']:<14.2f} ${lstm_results['rmse']:<14.2f} {lstm_results['r2']:<10.4f}")

## 6. Pillar 3: Reinforcement Learning — PPO Agent

### ★ STUDENT TODO: Design the reward function and train PPO

The RL agent learns a charging policy through trial and error. Unlike the LP
optimizer, it has **no knowledge of future prices** — it must learn to anticipate
patterns from experience.

**Your task:**
1. Review the reward function in `environment.py` (`step()` method, Section 8)
2. Modify reward weights in `config.py` or redesign the function
3. Train the PPO agent and analyze the learned policy
4. Compare RL vs. LP optimal — what is the "cost of uncertainty"?

In [ ]:
from rl_agent import (
    train_rl_agent,
    evaluate_rl_agent,
    run_single_episode,
    plot_training_curves,
    HAS_SB3,
)

if not HAS_SB3:
    print("⚠️  stable-baselines3 not installed. Skipping RL training.")
    print("   Install with: pip install stable-baselines3")

In [ ]:
if HAS_SB3:
    # Train PPO agent (reduce timesteps for quick iteration; increase for better policies)
    cfg.rl.total_timesteps = 100_000

    rl_result = train_rl_agent(cfg, verbose=True)

In [ ]:
if HAS_SB3:
    # Plot training progress (should see cost decreasing, targets met increasing)
    plot_training_curves(rl_result["training_stats"])

In [ ]:
if HAS_SB3:
    # Evaluate on the same scenario as heuristics/LP for fair comparison
    rl_eval = evaluate_rl_agent(
        rl_result["model"], cfg,
        schedules=schedules, price_curve=price_curve,
        n_episodes=5,
    )

    # Run single episode for detailed visualization
    rl_episode = run_single_episode(
        rl_result["model"], cfg,
        schedules=schedules, price_curve=price_curve,
    )
    plot_episode_results(rl_episode["env"], title="PPO RL Agent")

## 7. Comprehensive Strategy Comparison

In [ ]:
# Collect all results into a comparison table
comparison = []

# Heuristics
for h_result in heuristic_results:
    comparison.append(h_result)

# LP Optimal
comparison.append({
    "strategy": "lp_optimal",
    "net_cost": lp_result["net_cost"],
    "total_cost": lp_result["total_cost"],
    "v2g_revenue": lp_result["v2g_revenue"],
    "degradation_cost": lp_result.get("degradation_cost", 0),
    "penalties": 0,  # LP satisfies all constraints
    "evs_meeting_target": lp_result["evs_meeting_target"],
    "total_evs": lp_result["total_evs"],
})

# RL Agent (if trained)
if HAS_SB3:
    comparison.append(rl_episode)

# Display comparison table
comp_df = pd.DataFrame(comparison)
display_cols = ["strategy", "net_cost", "v2g_revenue", "penalties",
                "evs_meeting_target", "total_evs"]
existing_cols = [c for c in display_cols if c in comp_df.columns]
print("\n" + "="*80)
print("STRATEGY COMPARISON DASHBOARD")
print("="*80)
print(comp_df[existing_cols].to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

strategies = comp_df["strategy"].values
costs = comp_df["net_cost"].values

# Panel 1: Net Cost
ax1 = axes[0]
colors = ['#94A3B8', '#94A3B8', '#94A3B8', '#0D9488', '#F59E0B'][:len(strategies)]
bars = ax1.bar(strategies, costs, color=colors)
ax1.set_ylabel("Net Cost ($)")
ax1.set_title("Net Electricity Cost by Strategy")
ax1.tick_params(axis='x', rotation=45)
for bar, cost in zip(bars, costs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"${cost:.1f}", ha='center', va='bottom', fontsize=9)
ax1.grid(True, axis='y', alpha=0.3)

# Panel 2: EVs Meeting Target
ax2 = axes[1]
targets = comp_df["evs_meeting_target"].values
total = comp_df["total_evs"].values[0]
ax2.bar(strategies, targets, color=colors)
ax2.axhline(y=total, color='red', linestyle='--', label=f'Total EVs ({total})')
ax2.set_ylabel("EVs Meeting Target")
ax2.set_title("Departure Deadline Compliance")
ax2.tick_params(axis='x', rotation=45)
ax2.legend()
ax2.grid(True, axis='y', alpha=0.3)

# Panel 3: V2G Revenue
ax3 = axes[2]
if "v2g_revenue" in comp_df.columns:
    v2g = comp_df["v2g_revenue"].values
    ax3.bar(strategies, v2g, color=colors)
    ax3.set_ylabel("V2G Revenue ($)")
    ax3.set_title("Vehicle-to-Grid Revenue")
    ax3.tick_params(axis='x', rotation=45)
    ax3.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("strategy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Pillar 4: Agentic AI — LLM Orchestrator

The agentic AI uses an LLM (GPT-4o) with **tool-calling** to orchestrate all modules.
It provides a natural-language interface for fleet operators.

**Two modes:**
- **Live mode:** Set `OPENAI_API_KEY` environment variable to use real GPT-4o
- **Mock mode (default):** Works without any API key — demonstrates the full agentic pattern

To enable live mode:
```python
import os
os.environ["OPENAI_API_KEY"] = "sk-..."
```

In [ ]:
from agentic_ai import create_agent

# Get the best ML model for the agent's toolbox
best_ml_model = ml_models["models"].get("xgboost", None)

# Create the agent (auto-detects mock vs live mode based on API key)
agent = create_agent(
    cfg=cfg,
    schedules=schedules,
    price_curve=price_curve,
    ml_model=best_ml_model,
    lstm_dict=lstm_dict if 'lstm_dict' in dir() else None,
    rl_model=rl_result["model"] if HAS_SB3 and 'rl_result' in dir() else None,
    prices_df=prices_df,
)

In [ ]:
# Demo: Ask the agent about the cheapest charging plan
response = agent.chat("What's the cheapest charging plan for tonight?")
print(response)

In [ ]:
# Demo: Compare all strategies
response = agent.chat("Compare all strategies and tell me which is best.")
print(response)

In [ ]:
# Demo: V2G advice
response = agent.chat("Should I use V2G to sell energy during the evening peak?")
print(response)

## 9. ★ Analysis Report (Student Deliverable)

### Write your analysis here

Answer the following questions based on your experiments:

**1. ML vs. DL Forecasting**
- Which model achieved lower MAE? By how much?
- Which features were most important for XGBoost?
- Does the LSTM provide advantages over hand-crafted tabular features?
- When would you recommend ML vs. DL for price forecasting?

**2. RL vs. LP Optimal**
- What was the "cost of uncertainty" (gap between RL and LP)?
- How did your reward function design affect the learned policy?
- Did the RL agent learn to exploit V2G? How did its strategy compare to LP?
- What were the main failure modes of the RL agent?

**3. Overall Comparison**
- Rank all strategies by net cost. Does this ranking change if you weight constraint violations?
- Which approach would you deploy in a real depot? Why?
- What is the value of the agentic AI layer for production use?

**4. Extensions (optional)**
- Did you try any extensions (new features, different RL algorithms, etc.)? What did you learn?

In [ ]:
# YOUR ANALYSIS CODE AND DISCUSSION HERE
# Add as many cells as needed.

print("Analysis complete!")